In [1]:
print("all ok")

all ok


https://github.com/confident-ai/deepeval

https://deepeval.com/

https://www.confident-ai.com/

In [2]:
#RAG Evaluation
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [ ]:
# Agent Evaluation
from deepeval.metrics import (
    # Reasoning / Planning
    PlanQualityMetric,
    PlanAdherenceMetric,

    # Tool / Action
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,

    # Overall Execution
    TaskCompletionMetric,
    StepEfficiencyMetric,
)

from deepeval.metrics import (
    ToolUseMetric,
    GoalAccuracyMetric,
)

from deepeval.metrics import (
    ConversationCompletenessMetric,
    ConversationRelevancyMetric,
    RoleAdherenceMetric,
    KnowledgeRetentionMetric,
)

In [4]:
# Safety Evaluation
from deepeval.metrics import (
    BiasMetric,
    ToxicityMetric,
    NonAdviceMetric,
    MisuseMetric,
    PIILeakageMetric,
    RoleViolationMetric,
)

In [5]:
# General LLM / Output Evaluation
from deepeval.metrics import (
    HallucinationMetric,
    PromptAlignmentMetric,
    JsonCorrectnessMetric,
    SummarizationMetric,
    ExactMatchMetric,
)

In [6]:
# Custom LLM-as-a-Judge
from deepeval.metrics import (
    GEval,
    DAGMetric,
    ArenaGEval,
)

In [7]:
from deepeval.test_case import (
    LLMTestCase,
    ToolCall,
    SingleTurnParams,
)
from deepeval.test_case import (
    ConversationalTestCase,
    Turn,
)
from deepeval.test_case import (
    ArenaTestCase,
    Contestant,
)
from deepeval.dataset import (
    EvaluationDataset,
    Golden,
)
from deepeval import (
    evaluate,
    assert_test,
)
from deepeval.tracing import (
    observe,
    update_current_span,
)

In [ ]:
# from deepeval import evaluate, assert_test

# from deepeval.dataset import (
#     EvaluationDataset,
#     Golden,
# )

# from deepeval.test_case import (
#     LLMTestCase,
#     ConversationalTestCase,
#     Turn,
#     ToolCall,
#     SingleTurnParams,
#     ArenaTestCase,
#     Contestant,
# )

# from deepeval.metrics import (
#     # RAG
#     AnswerRelevancyMetric,
#     FaithfulnessMetric,
#     ContextualRelevancyMetric,
#     ContextualPrecisionMetric,
#     ContextualRecallMetric,

#     # Agent
#     PlanQualityMetric,
#     PlanAdherenceMetric,
#     ToolCorrectnessMetric,
#     ArgumentCorrectnessMetric,
#     TaskCompletionMetric,
#     StepEfficiencyMetric,
#     ToolUseMetric,
#     GoalAccuracyMetric,

#     # Chatbot
#     ConversationCompletenessMetric,
#     ConversationRelevancyMetric,
#     RoleAdherenceMetric,
#     KnowledgeRetentionMetric,

#     # Safety
#     BiasMetric,
#     ToxicityMetric,
#     NonAdviceMetric,
#     MisuseMetric,
#     PIILeakageMetric,
#     RoleViolationMetric,

#     # General
#     HallucinationMetric,
#     PromptAlignmentMetric,
#     JsonCorrectnessMetric,
#     SummarizationMetric,
#     ExactMatchMetric,

#     # Custom
#     GEval,
#     DAGMetric,
#     ArenaGEval,
# )

# from deepeval.tracing import (
#     observe,
#     update_current_span,
# )

In [9]:
import os
import asyncio
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.vectorstores import InMemoryVectorStore

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY is missing. Please add it to the .env file."
    )

In [11]:
documents = [
    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={"id": "ragas_intro"},
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications.

Experiments can track evaluation results,
compare different application versions,
and store experiment outputs.
""".strip(),
        metadata={"id": "ragas_experiments"},
    ),

    Document(
        page_content="""
Ragas metrics measure different properties
of AI systems.

Examples include factual correctness,
faithfulness, answer relevancy,
retrieval quality, and agent behavior.
""".strip(),
        metadata={"id": "ragas_metrics"},
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge source and provides that
information to an LLM before generating
the final answer.
""".strip(),
        metadata={"id": "rag_intro"},
    ),
]

In [12]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

In [13]:
vector_store = InMemoryVectorStore(
    embedding=embedding_model,
)

vector_store.add_documents(documents)

['a51c0dbb-dd53-4bc6-a1c4-7faaeed56864',
 'cec19349-f417-483b-85e7-bdcdd8b3587a',
 '898e20e6-f560-4d83-8851-2711d97c0bcd',
 '4c4292c3-f680-42ec-a310-1714ae0df993']

In [14]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
)

In [15]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [16]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question using the provided context.

If the context does not contain enough information,
say that you do not have enough information.

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [18]:
rag_chain = (
    rag_prompt
    | llm
    | StrOutputParser()
)

In [19]:
async def rag_query(question):

    retrieved_documents = await retriever.ainvoke(question)

    retrieved_contexts = [
        document.page_content
        for document in retrieved_documents
    ]

    retrieved_document_ids = [
        document.metadata["id"]
        for document in retrieved_documents
    ]

    context = "\n\n".join(retrieved_contexts)

    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    return {
        "answer": answer,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,
    }

In [20]:
question = "What is RAG and how does it work?"

In [22]:
await rag_query(question)

{'answer': "RAG stands for Retrieval-Augmented Generation. It is a system that retrieves relevant information from a knowledge source and provides that information to a large language model (LLM) before generating the final answer. This process enhances the LLM's ability to produce accurate and contextually relevant responses by incorporating external information.",
 'retrieved_contexts': ['RAG stands for Retrieval-Augmented Generation.\n\nA RAG system retrieves relevant information\nfrom a knowledge source and provides that\ninformation to an LLM before generating\nthe final answer.',
  'Ragas is an open-source evaluation framework designed\nfor evaluating and improving LLM applications.\n\nIt supports RAG evaluation, agent evaluation,\ncustom metrics, datasets, experiments,\nand synthetic test data generation.'],
 'retrieved_document_ids': ['rag_intro', 'ragas_intro']}

## Evaluation of the above app

In [23]:
# ============================================================
# 11. LOAD EVALUATION DATASET
# ============================================================

def load_dataset():

    return [
        {
            "question": "What is Ragas?",
            "expected_answer": """
Ragas is an evaluation framework
for LLM applications.
""".strip(),
        },

        {
            "question": "How do Ragas experiments work?",
            "expected_answer": """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),
        },

        {
            "question": "What do Ragas metrics do?",
            "expected_answer": """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),
        },

        {
            "question": "What is RAG?",
            "expected_answer": """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),
        },
    ]

In [24]:
load_dataset()

[{'question': 'What is Ragas?',
  'expected_answer': 'Ragas is an evaluation framework\nfor LLM applications.'},
 {'question': 'How do Ragas experiments work?',
  'expected_answer': 'Ragas experiments provide systematic evaluation,\ntrack results, compare different application versions,\nand store experiment outputs.'},
 {'question': 'What do Ragas metrics do?',
  'expected_answer': 'Ragas metrics evaluate properties and quality\nof AI or LLM application outputs.'},
 {'question': 'What is RAG?',
  'expected_answer': 'RAG stands for Retrieval-Augmented Generation.\n\nIt retrieves relevant information from a knowledge\nsource and provides that context to an LLM\nbefore answer generation.'}]

In [26]:
from deepeval.models import OpenAIModel

In [27]:
judge_model = OpenAIModel(model="gpt-5.6-terra")

In [28]:

def create_metrics():

    answer_relevancy = AnswerRelevancyMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    faithfulness = FaithfulnessMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_relevancy = ContextualRelevancyMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_precision = ContextualPrecisionMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    contextual_recall = ContextualRecallMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
    )

    return {
        "answer_relevancy": answer_relevancy,
        "faithfulness": faithfulness,
        "contextual_relevancy": contextual_relevancy,
        "contextual_precision": contextual_precision,
        "contextual_recall": contextual_recall,
    }

In [29]:
async def evaluate_test_case(row):

    question = row["question"]
    expected_answer = row["expected_answer"]

    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    rag_result = await rag_query(question)

    response = rag_result["answer"]
    retrieved_contexts = rag_result["retrieved_contexts"]
    retrieved_document_ids = rag_result["retrieved_document_ids"]

    # ========================================================
    # CREATE DEEPEVAL TEST CASE
    # ========================================================

    test_case = LLMTestCase(
        input=question,
        actual_output=response,
        expected_output=expected_answer,
        retrieval_context=retrieved_contexts,
    )

    # ========================================================
    # CREATE METRICS
    # ========================================================

    metrics = create_metrics()

    # ========================================================
    # ANSWER RELEVANCY
    # ========================================================

    await metrics["answer_relevancy"].a_measure(test_case)

    # ========================================================
    # FAITHFULNESS
    # ========================================================

    await metrics["faithfulness"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL RELEVANCY
    # ========================================================

    await metrics["contextual_relevancy"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL PRECISION
    # ========================================================

    await metrics["contextual_precision"].a_measure(test_case)

    # ========================================================
    # CONTEXTUAL RECALL
    # ========================================================

    await metrics["contextual_recall"].a_measure(test_case)

    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {
        "question": question,
        "expected_answer": expected_answer,
        "response": response,
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,

        "answer_relevancy": metrics["answer_relevancy"].score,
        "answer_relevancy_reason": metrics["answer_relevancy"].reason,

        "faithfulness": metrics["faithfulness"].score,
        "faithfulness_reason": metrics["faithfulness"].reason,

        "contextual_relevancy": metrics["contextual_relevancy"].score,
        "contextual_relevancy_reason": metrics["contextual_relevancy"].reason,

        "contextual_precision": metrics["contextual_precision"].score,
        "contextual_precision_reason": metrics["contextual_precision"].reason,

        "contextual_recall": metrics["contextual_recall"].score,
        "contextual_recall_reason": metrics["contextual_recall"].reason,
    }

In [30]:
async def run_deepeval_evaluation(dataset):

    results = []

    for row in tqdm(
        dataset,
        desc="Running DeepEval evaluation",
    ):

        result = await evaluate_test_case(row)

        results.append(result)

    return results

In [31]:
# ============================================================
# 16. MAIN FUNCTION
# ============================================================

async def main():

    print("\n" + "=" * 100)
    print("DEEPEVAL + LANGCHAIN RAG EVALUATION")
    print("=" * 100)

    # ========================================================
    # LOAD DATASET
    # ========================================================

    dataset = load_dataset()

    print("\nDataset loaded.")

    # ========================================================
    # RUN DEEPEVAL EVALUATION
    # ========================================================

    results = await run_deepeval_evaluation(dataset)

    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "answer_relevancy",
        "faithfulness",
        "contextual_relevancy",
        "contextual_precision",
        "contextual_recall",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CALCULATE AVERAGE SCORES
    # ========================================================

    average_answer_relevancy = df["answer_relevancy"].mean()
    average_faithfulness = df["faithfulness"].mean()
    average_contextual_relevancy = df["contextual_relevancy"].mean()
    average_contextual_precision = df["contextual_precision"].mean()
    average_contextual_recall = df["contextual_recall"].mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Answer Relevancy      : {average_answer_relevancy:.2f}")
    print(f"Average Faithfulness          : {average_faithfulness:.2f}")
    print(f"Average Contextual Relevancy  : {average_contextual_relevancy:.2f}")
    print(f"Average Contextual Precision  : {average_contextual_precision:.2f}")
    print(f"Average Contextual Recall     : {average_contextual_recall:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    experiment_directory = Path("evals/deepeval")
    experiment_directory.mkdir(parents=True, exist_ok=True)

    output_file = experiment_directory / "deepeval_langchain_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nRESPONSE:")
        print(result_row["response"])

        print("\nRETRIEVED DOCUMENT IDS:")
        print(result_row["retrieved_document_ids"])

        print("\nANSWER RELEVANCY:")
        print(result_row["answer_relevancy"])

        print("\nANSWER RELEVANCY REASON:")
        print(result_row["answer_relevancy_reason"])

        print("\nFAITHFULNESS:")
        print(result_row["faithfulness"])

        print("\nFAITHFULNESS REASON:")
        print(result_row["faithfulness_reason"])

        print("\nCONTEXTUAL RELEVANCY:")
        print(result_row["contextual_relevancy"])

        print("\nCONTEXTUAL RELEVANCY REASON:")
        print(result_row["contextual_relevancy_reason"])

        print("\nCONTEXTUAL PRECISION:")
        print(result_row["contextual_precision"])

        print("\nCONTEXTUAL PRECISION REASON:")
        print(result_row["contextual_precision_reason"])

        print("\nCONTEXTUAL RECALL:")
        print(result_row["contextual_recall"])

        print("\nCONTEXTUAL RECALL REASON:")
        print(result_row["contextual_recall_reason"])

        print("\n" + "-" * 100)

In [32]:
await main()


DEEPEVAL + LANGCHAIN RAG EVALUATION

Dataset loaded.


Running DeepEval evaluation:   0%|          | 0/4 [00:00<?, ?it/s]

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<string>:4: RuntimeWarning: coroutine 'rag_query' was never awaited
RuntimeWarning: Enable tracemalloc to get the object allocation traceback

Running DeepEval evaluation:  25%|██▌       | 1/4 [00:23<01:10, 23.53s/it]

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

... (2 outputs left out)

Running DeepEval evaluation:  75%|███████▌  | 3/4 [01:01<00:19, 19.77s/it]

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Running DeepEval evaluation: 100%|██████████| 4/4 [01:19<00:00, 19.80s/it]


DETAILED RESULTS
                      question  answer_relevancy  faithfulness  contextual_relevancy  contextual_precision  contextual_recall
                What is Ragas?               1.0           1.0                   1.0                   1.0                1.0
How do Ragas experiments work?               1.0           1.0                   1.0                   1.0                1.0
     What do Ragas metrics do?               1.0           1.0                   1.0                   1.0                1.0
                  What is RAG?               1.0           1.0                   1.0                   1.0                1.0

SUMMARY
Average Answer Relevancy      : 1.00
Average Faithfulness          : 1.00
Average Contextual Relevancy  : 1.00
Average Contextual Precision  : 1.00
Average Contextual Recall     : 1.00

Results saved to:
evals\deepeval\deepeval_langchain_results.csv

DETAILED EVALUATION REASONS

QUESTION:
What is Ragas?

RESPONSE:
Ragas is an open-source eva